# Team 02 · FM 최소 프로토타입
규모·모멘텀 두 변수로 내부 검증을 실행합니다. 최종 연구 모형은 아닙니다.
모멘텀: 입력월 t−12~t−2(11개월). 테스트 데이터는 사용하지 않습니다.
Supabase 서명 URL은 팀 내부에서 전달받고, 공개 노트북 셀에 적지 마세요.
셀을 위에서 아래로 실행합니다. 현재 런타임에서 이미 다른 pandas/numpy를 사용했다면 패키지 설치 후 런타임을 다시 시작하세요.


In [ ]:
%pip install -q pandas==2.2.3 numpy==2.3.5


In [ ]:
from pathlib import Path
import os
WORK=Path("/content/fdi_team02")
WORK.mkdir(exist_ok=True)
os.chdir(WORK)
SOURCES={'load_training_data.py': '"""Load only the supplied warmup/train CSVs. Never discover or read test files."""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport io\nimport json\nimport platform\nimport re\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nPERIODS = {\n    "warmup_2000": ("2000-01", "2000-12"),\n    "train_2001_2021": ("2001-01", "2021-12"),\n}\nNUMERIC = (\n    "adjusted_monthly_return", "raw_month_end_price",\n    "raw_month_end_shares_outstanding", "raw_month_end_market_cap",\n)\nVARIABLES = NUMERIC + ("isin", "company_name")\n\n\n@dataclass\nclass LoadResult:\n    panel: pd.DataFrame\n    audit: dict\n\n\ndef _read_wide(path: Path, period: str, variable: str, expected_count: int):\n    # Exact allowlist filenames only; no glob and no test-period parameter.\n    expected_name = f"{period}__{variable}.csv"\n    if period not in PERIODS or variable not in VARIABLES or path.name != expected_name:\n        raise ValueError("허용되지 않은 입력 파일")\n    if path.is_symlink():\n        raise ValueError(f"심볼릭 링크 입력은 허용하지 않습니다: {path.name}")\n    raw = path.read_bytes()\n    rows = list(csv.reader(io.StringIO(raw.decode("utf-8-sig"))))\n    if not rows or not rows[0] or rows[0][0] != "month":\n        raise ValueError(f"month 헤더 없음: {path.name}")\n    header = rows[0]\n    codes = header[1:]\n    if len(codes) != len(set(codes)) or not codes:\n        raise ValueError(f"중복 또는 빈 종목코드: {path.name}")\n    if len(codes) != expected_count:\n        raise ValueError(f"종목 수 불일치: {path.name}: {len(codes)} != {expected_count}")\n    if any(re.fullmatch(r"[0-9A-Z]{6}", code) is None for code in codes):\n        raise ValueError(f"6자리 종목코드 형식 오류: {path.name}")\n    if any(len(row) != len(header) for row in rows[1:]):\n        raise ValueError(f"행 길이 불일치: {path.name}")\n    frame = pd.DataFrame(rows[1:], columns=header, dtype="string")\n    start, end = PERIODS[period]\n    expected_months = pd.period_range(start, end, freq="M").astype(str).tolist()\n    if frame["month"].tolist() != expected_months:\n        raise ValueError(f"월 누락·중복·순서·기간 오류: {path.name}")\n    wide = frame.set_index("month").replace("", pd.NA)\n    wide.columns.name = "stock_code"\n    if variable in NUMERIC:\n        try:\n            wide = wide.apply(pd.to_numeric, errors="raise").astype(float)\n        except (ValueError, TypeError) as exc:\n            raise ValueError(f"비수치 데이터: {path.name}") from exc\n        if np.isinf(wide.to_numpy()).any():\n            raise ValueError(f"무한대 데이터: {path.name}")\n    series = wide.stack(future_stack=True).rename(variable)\n    record = {\n        "filename": path.name, "sha256": hashlib.sha256(raw).hexdigest(),\n        "bytes": len(raw), "months": len(wide), "stocks": len(codes),\n        "missing": int(series.isna().sum()),\n    }\n    return series, codes, record\n\n\ndef load_training_data(data_dir: str | Path, expected_stock_count: int = 1086) -> LoadResult:\n    """Return every month/code cell, preserving missing values and string codes.\n\n    month is an ISO YYYY-MM string. adjusted_monthly_return is contemporaneous\n    raw input, NOT the prediction target. No feature imputation or scaling occurs.\n    """\n    data_dir = Path(data_dir)\n    if expected_stock_count < 1:\n        raise ValueError("expected_stock_count는 양수여야 합니다.")\n    frames, files, reference_codes = [], [], None\n    for period in PERIODS:\n        columns = []\n        for variable in VARIABLES:\n            series, codes, record = _read_wide(\n                data_dir / f"{period}__{variable}.csv", period, variable, expected_stock_count\n            )\n            if reference_codes is None:\n                reference_codes = codes\n            elif codes != reference_codes:\n                raise ValueError(f"파일 간 종목코드 또는 순서 불일치: {record[\'filename\']}")\n            columns.append(series)\n            files.append(record)\n        frame = pd.concat(columns, axis=1).reset_index()\n        frame["source_period"] = period\n        frames.append(frame)\n    panel = pd.concat(frames, ignore_index=True).sort_values(\n        ["month", "stock_code"], kind="stable"\n    ).reset_index(drop=True)\n    panel["stock_code"] = panel["stock_code"].astype("string")\n    if panel.duplicated(["month", "stock_code"]).any():\n        raise ValueError("종목·월 중복 키")\n    panel["has_market_observation"] = panel[list(NUMERIC)].notna().any(axis=1)\n    observed = panel[panel["has_market_observation"]]\n    warnings = []\n    for variable in NUMERIC[1:]:\n        count = int((panel[variable] <= 0).sum())\n        if count:\n            warnings.append({"kind": "nonpositive_raw_value", "column": variable, "rows": count})\n    below_minus_one = int((panel["adjusted_monthly_return"] < -1).sum())\n    if below_minus_one:\n        warnings.append({"kind": "return_below_minus_one", "rows": below_minus_one})\n    audit = {\n        "schema_version": "0.1", "python": platform.python_version(),\n        "pandas": pd.__version__, "numpy": np.__version__,\n        "files_read": files, "allowed_periods": list(PERIODS),\n        "rows": len(panel), "months": panel["month"].nunique(),\n        "stocks": panel["stock_code"].nunique(),\n        "month_min": panel["month"].min(), "month_max": panel["month"].max(),\n        "rows_with_market_observation": len(observed),\n        "missing_by_variable": {v: int(panel[v].isna().sum()) for v in VARIABLES},\n        "observed_rows_by_month": observed.groupby("month").size().to_dict(),\n        "warnings": warnings,\n        "notes": [\n            "전 기간 종목 집합의 모든 월·종목 조합을 보존하며 결측 행을 제거하지 않음",\n            "has_market_observation은 실제 거래가능성·상장여부의 완전한 판정이 아님",\n            "수익률·특성 결측을 0이나 이전 값으로 채우지 않음",\n            "테스트 파일을 열지 않으며 재무정보·텍스트는 결합하지 않음",\n        ],\n    }\n    return LoadResult(panel, audit)\n\n\ndef make_training_targets(panel: pd.DataFrame) -> pd.DataFrame:\n    """Separate labels from features; join exact calendar t+1, never next observation.\n\n    Labels only span 2001-01..2021-12. Rank denominator is all nonmissing\n    supplied returns that month, before any later model-specific sample filter.\n    """\n    if panel.duplicated(["month", "stock_code"]).any():\n        raise ValueError("종목·월 중복 키")\n    if not panel["month"].between("2000-01", "2021-12").all():\n        raise ValueError("학습/준비 기간 밖의 패널은 허용하지 않습니다.")\n    left = panel[["month", "stock_code"]].copy()\n    left["target_month"] = (pd.PeriodIndex(left["month"], freq="M") + 1).astype(str)\n    left = left[left["target_month"].between("2001-01", "2021-12")]\n    right = panel[["month", "stock_code", "adjusted_monthly_return"]].rename(columns={\n        "month": "target_month", "adjusted_monthly_return": "target_return"\n    })\n    right = right[right["target_month"].between("2001-01", "2021-12")].copy()\n    right["target_rank"] = right.groupby("target_month")["target_return"].rank(\n        method="average", pct=True, na_option="keep"\n    )\n    targets = left.merge(right, how="left", on=["target_month", "stock_code"], validate="one_to_one")\n    targets["has_target"] = targets["target_return"].notna()\n    return targets.sort_values(["month", "stock_code"]).reset_index(drop=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description="준비용·학습용 12개 CSV만 읽는 패널 로더")\n    parser.add_argument("--data-dir", required=True, type=Path)\n    parser.add_argument("--output-dir", required=True, type=Path)\n    parser.add_argument("--config", type=Path, default=Path(__file__).with_name("config.json"))\n    parser.add_argument("--with-targets", action="store_true", help="다음 달 학습 목표를 별도 CSV로 저장")\n    args = parser.parse_args()\n    config = json.loads(args.config.read_text(encoding="utf-8"))\n    result = load_training_data(args.data_dir, config["expected_stock_count"])\n    output_names = ["market_panel_train.csv.gz", "load_audit.json"]\n    if args.with_targets:\n        output_names.append("training_targets.csv.gz")\n    if any((args.output_dir / name).exists() for name in output_names):\n        parser.error("기존 산출물이 있습니다. 새로운 --output-dir을 사용하세요.")\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    result.panel.to_csv(args.output_dir / output_names[0], index=False, encoding="utf-8", compression="gzip")\n    if args.with_targets:\n        targets = make_training_targets(result.panel)\n        targets.to_csv(args.output_dir / "training_targets.csv.gz", index=False, encoding="utf-8", compression="gzip")\n        result.audit["targets"] = {\n            "rows": len(targets), "nonmissing": int(targets["has_target"].sum()),\n            "target_month_min": targets["target_month"].min(),\n            "target_month_max": targets["target_month"].max(),\n            "rank_scope": "각 목표월의 제공 수익률 비결측 종목 전체, 평균 동순위 / 관측수",\n        }\n    (args.output_dir / "load_audit.json").write_text(\n        json.dumps(result.audit, ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    print(f"완료: 허용 파일 {len(result.audit[\'files_read\'])}개, "\n          f"{result.audit[\'months\']}개월, {result.audit[\'stocks\']:,}종목, {len(result.panel):,}행")\n    print(f"저장 위치: {args.output_dir}")\n\n\nif __name__ == "__main__":\n    main()\n', 'fm_pipeline.py': '"""Small, deterministic characteristic-based FM prototype; training data only."""\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nFEATURES = ["size_rank", "momentum_rank"]\nCOEFFICIENTS = ["intercept", *FEATURES]\n\n\ndef build_features(panel: pd.DataFrame, momentum: dict) -> pd.DataFrame:\n    if momentum.get("status") != "confirmed":\n        raise ValueError("모멘텀 기간을 팀 확인 후 config.json에 확정하세요.")\n    start = momentum.get("start_lag_from_target_month")\n    end = momentum.get("end_lag_from_target_month")\n    if not isinstance(start, int) or not isinstance(end, int) or not start >= end >= 1:\n        raise ValueError("모멘텀 시차는 예측월 기준 start >= end >= 1이어야 합니다.")\n    if panel.duplicated(["month", "stock_code"]).any():\n        raise ValueError("중복 종목·월")\n    returns = panel.pivot(index="month", columns="stock_code", values="adjusted_monthly_return")\n    months = pd.period_range(returns.index.min(), returns.index.max(), freq="M").astype(str)\n    returns = returns.reindex(months)\n    # Invalid returns and missing history propagate to the entire window.\n    returns = returns.where(returns >= -1)\n    cumulative = pd.DataFrame(1.0, index=returns.index, columns=returns.columns)\n    for lag in range(end, start + 1):\n        # Input month t = target month m - 1; m-lag = t-(lag-1).\n        cumulative = cumulative * (1 + returns.shift(lag - 1))\n    cumulative = cumulative - 1\n    cumulative.index.name = "month"\n    cumulative.columns.name = "stock_code"\n    mom = cumulative.stack(future_stack=True).rename("momentum_raw").reset_index()\n    out = panel[["month", "stock_code", "raw_month_end_market_cap"]].merge(\n        mom, on=["month", "stock_code"], validate="one_to_one"\n    )\n    out["eligible_at_input"] = out.raw_month_end_market_cap.gt(0)\n    out["size_raw"] = np.log(out.raw_month_end_market_cap.where(out.eligible_at_input))\n    out["momentum_missing"] = out.momentum_raw.isna()\n    # Candidate universe depends only on input-month information, never future labels.\n    for raw, ranked in [("size_raw", "size_rank"), ("momentum_raw", "momentum_rank")]:\n        values = out[raw].where(out.eligible_at_input)\n        out[ranked] = values.groupby(out.month).rank(method="average", pct=True) - 0.5\n        out.loc[out.eligible_at_input, ranked] = out.loc[out.eligible_at_input, ranked].fillna(0)\n    return out\n\n\ndef fit_monthly_fm(data: pd.DataFrame, train_start: str, train_end: str, min_stocks: int = 30):\n    train = data[data.target_month.between(train_start, train_end)]\n    rows, logs = [], []\n    for month in pd.period_range(train_start, train_end, freq="M").astype(str):\n        group = train[(train.target_month == month) & train.eligible_at_input].dropna(\n            subset=["target_rank", *FEATURES]\n        )\n        log = {"target_month": month, "n_stocks": len(group), "status": "ok"}\n        if len(group) < max(min_stocks, len(COEFFICIENTS) + 1):\n            log["status"] = "insufficient_stocks"\n        else:\n            x = np.column_stack([np.ones(len(group)), group[FEATURES].to_numpy(float)])\n            if np.linalg.matrix_rank(x) < x.shape[1]:\n                log["status"] = "rank_deficient"\n            else:\n                beta, _, _, _ = np.linalg.lstsq(x, group.target_rank.to_numpy(float), rcond=None)\n                rows.append({"target_month": month, **dict(zip(COEFFICIENTS, beta))})\n        logs.append(log)\n    if not rows:\n        raise ValueError("적합 가능한 학습월이 없습니다.")\n    monthly = pd.DataFrame(rows)\n    mean = monthly[COEFFICIENTS].mean().to_dict()\n    return monthly, mean, pd.DataFrame(logs)\n\n\ndef predict(features: pd.DataFrame, coefficients: dict) -> pd.Series:\n    # Explicit whitelist prevents accidental inclusion of target columns.\n    return coefficients["intercept"] + features[FEATURES].mul(\n        pd.Series({name: coefficients[name] for name in FEATURES})\n    ).sum(axis=1, min_count=len(FEATURES))\n\n\ndef monthly_ic(predictions: pd.DataFrame, start: str, end: str, min_stocks: int = 30):\n    rows = []\n    for month in pd.period_range(start, end, freq="M").astype(str):\n        group = predictions[predictions.target_month == month].dropna(subset=["prediction", "target_return"])\n        status, ic = "ok", np.nan\n        if len(group) < min_stocks:\n            status = "insufficient_stocks"\n        elif group.prediction.nunique() < 2 or group.target_return.nunique() < 2:\n            status = "constant_rank"\n        else:\n            # Spearman = Pearson correlation of average ranks; no scipy dependency.\n            ic = group.prediction.rank(method="average").corr(group.target_return.rank(method="average"))\n        rows.append({"target_month": month, "n_stocks": len(group), "ic": ic, "status": status})\n    return pd.DataFrame(rows)\n\n\ndef evaluate_fold(data: pd.DataFrame, fold: dict, min_stocks: int = 30):\n    train_start, train_end = fold["train"]\n    val_start, val_end = fold["validation"]\n    if not train_start <= train_end < val_start <= val_end <= "2021-12":\n        raise ValueError("학습/검증 기간이 겹치거나 허용 범위를 벗어납니다.")\n    monthly, mean, log = fit_monthly_fm(data, train_start, train_end, min_stocks)\n    val = data[data.target_month.between(val_start, val_end) & data.eligible_at_input].copy()\n    val["prediction"] = predict(val, mean)\n    scores = monthly_ic(val, val_start, val_end, min_stocks)\n    return monthly, mean, log, val, scores\n', 'run_all.py': '"""Run warmup/train loading, features, expanding validation and final train fit."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport pandas as pd\n\nfrom load_training_data import load_training_data, make_training_targets\nfrom fm_pipeline import FEATURES, build_features, evaluate_fold, fit_monthly_fm\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--data-dir", type=Path, required=True)\n    p.add_argument("--output-dir", type=Path, required=True)\n    p.add_argument("--config", type=Path, default=Path(__file__).with_name("config.json"))\n    args = p.parse_args()\n    config = json.loads(args.config.read_text())\n    if config["momentum"]["status"] != "confirmed":\n        p.error("모멘텀 기간 미확정. 팀 확인 전 학습하지 않습니다.")\n    if args.output_dir.exists() and any(args.output_dir.iterdir()):\n        p.error("출력 폴더가 비어 있지 않습니다. 새 출력 폴더를 지정하세요.")\n    loaded = load_training_data(args.data_dir, config["expected_stock_count"])\n    features = build_features(loaded.panel, config["momentum"])\n    targets = make_training_targets(loaded.panel)\n    data = features.merge(targets, on=["month", "stock_code"], how="inner", validate="one_to_one")\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    (args.output_dir / "load_audit.json").write_text(json.dumps(loaded.audit, ensure_ascii=False, indent=2))\n    summaries, all_ic = [], []\n    for i, fold in enumerate(config["validation_folds_by_target_month"], 1):\n        directory = args.output_dir / f"fold_{i}"\n        directory.mkdir()\n        monthly, mean, logs, prediction, scores = evaluate_fold(data, fold)\n        monthly.to_csv(directory / "monthly_coefficients.csv", index=False)\n        logs.to_csv(directory / "fit_log.csv", index=False)\n        prediction[["month", "target_month", "stock_code", "prediction", "target_return", "target_rank"]].to_csv(\n            directory / "validation_predictions.csv.gz", index=False, compression="gzip"\n        )\n        scores.to_csv(directory / "monthly_ic.csv", index=False)\n        (directory / "mean_coefficients.json").write_text(json.dumps(mean, indent=2))\n        valid = scores.ic.dropna()\n        summaries.append({"fold": i, "train": fold["train"], "validation": fold["validation"],\n                          "fit_months": len(monthly), "skipped_fit_months": int(logs.status.ne("ok").sum()),\n                          "valid_ic_months": len(valid), "mean_ic": float(valid.mean()) if len(valid) else None,\n                          "positive_ic_fraction": float(valid.gt(0).mean()) if len(valid) else None})\n        all_ic.append(scores.assign(fold=i))\n    combined = pd.concat(all_ic, ignore_index=True)\n    if combined.target_month.duplicated().any():\n        raise ValueError("검증 목표월 중복: 전체 IC 집계 불가")\n    combined.to_csv(args.output_dir / "monthly_ic.csv", index=False)\n    valid = combined.ic.dropna()\n    summary = {"model": "FM size + momentum prototype", "features": FEATURES,\n               "momentum": config["momentum"], "folds": summaries,\n               "overall_mean_ic": float(valid.mean()) if len(valid) else None,\n               "valid_ic_months": len(valid), "test_evaluated": False,\n               "note": "학습기간 내부 검증 결과. 최종 모델 또는 텍스트 추가 효과의 검증 결과가 아님."}\n    (args.output_dir / "validation_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2))\n    monthly, mean, log = fit_monthly_fm(data, "2001-01", "2021-12")\n    frozen = {"training_target_months": ["2001-01", "2021-12"], "features": FEATURES,\n              "coefficients": mean, "config": config, "n_fit_months": len(monthly),\n              "input_hashes": {f["filename"]: f["sha256"] for f in loaded.audit["files_read"]},\n              "prototype_only": True}\n    payload = json.dumps(frozen, ensure_ascii=False, sort_keys=True, indent=2).encode()\n    (args.output_dir / "frozen_prototype.json").write_bytes(payload)\n    (args.output_dir / "frozen_prototype.sha256").write_text(hashlib.sha256(payload).hexdigest() + "\\n")\n    monthly.to_csv(args.output_dir / "full_train_monthly_coefficients.csv", index=False)\n    log.to_csv(args.output_dir / "full_train_fit_log.csv", index=False)\n    print(json.dumps(summary, ensure_ascii=False, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', 'download_training_data.py': '"""Download the private Supabase ZIP using a temporary signed URL."""\nimport argparse\nimport getpass\nimport hashlib\nimport io\nimport json\nimport urllib.parse\nimport urllib.request\nimport zipfile\nfrom pathlib import Path\nfrom load_training_data import PERIODS, VARIABLES\n\n\ndef download_training_data(signed_url, destination, manifest_path=None):\n    parsed = urllib.parse.urlsplit(signed_url)\n    if parsed.scheme != \'https\' or parsed.hostname != \'jiruebyoipsuzywedbhq.supabase.co\':\n        raise ValueError(\'이 프로젝트의 HTTPS Supabase 링크만 허용합니다.\')\n    manifest_path = manifest_path or Path(__file__).with_name(\'data_manifest.json\')\n    manifest = json.loads(Path(manifest_path).read_text())\n    with urllib.request.urlopen(signed_url, timeout=60) as response:\n        data = response.read(manifest[\'bytes\'] + 1)\n    if len(data) != manifest[\'bytes\'] or hashlib.sha256(data).hexdigest() != manifest[\'sha256\']:\n        raise ValueError(\'데이터 크기 또는 SHA-256 불일치\')\n    dest = Path(destination)\n    dest.mkdir(parents=True, exist_ok=True)\n    allowed = {f\'raw/{period}__{v}.csv\' for period in PERIODS for v in VARIABLES}\n    with zipfile.ZipFile(io.BytesIO(data)) as archive:\n        names = archive.namelist()\n        if len(names) != len(set(names)) or set(names) != allowed | {\'README.txt\'}:\n            raise ValueError(\'예상하지 않은 압축파일 구성\')\n        for name in sorted(allowed):\n            target = dest / Path(name).name\n            content = archive.read(name)\n            if target.exists() and target.read_bytes() != content:\n                raise ValueError(f\'기존 데이터 불일치: {target.name}\')\n        for name in sorted(allowed):\n            (dest / Path(name).name).write_bytes(archive.read(name))\n    return dest\n\n\nif __name__ == \'__main__\':\n    p=argparse.ArgumentParser()\n    p.add_argument(\'--output-dir\',type=Path,required=True)\n    args=p.parse_args()\n    url=getpass.getpass(\'Supabase 서명 다운로드 URL (입력 숨김): \')\n    download_training_data(url,args.output_dir)\n    print(\'학습 데이터 12개 다운로드 및 해시 검사 완료\')\n', 'config.json': '{\n  "expected_stock_count": 1086,\n  "momentum": {\n    "status": "confirmed",\n    "start_lag_from_target_month": 13,\n    "end_lag_from_target_month": 3,\n    "definition": "입력월 t의 t-12~t-2, 11개월. 예측월 m=t+1 기준 m-13~m-3.",\n    "confirmed_by_user": "2026-10-17"\n  },\n  "target": "next_month_return_percentile_rank",\n  "validation_folds_by_target_month": [\n    {"train": ["2001-01", "2012-12"], "validation": ["2013-01", "2015-12"]},\n    {"train": ["2001-01", "2015-12"], "validation": ["2016-01", "2018-12"]},\n    {"train": ["2001-01", "2018-12"], "validation": ["2019-01", "2021-12"]}\n  ]\n}\n', 'data_manifest.json': '{\n  "filename": "fdi-team02-training-v1.zip",\n  "sha256": "1a6e980f10fafc6b70754fd7d3ab9db7005b2210729806e5fca9d8e1e4006665",\n  "bytes": 2909460,\n  "files": 12\n}'}
for name,content in SOURCES.items():
    (WORK/name).write_text(content,encoding="utf-8")


## 비공개 학습 자료 다운로드
Colab Secret `FDI_TRAINING_URL` 또는 숨김 입력을 사용합니다. 링크가 만료되면 프로젝트 소유자에게 새 서명 URL을 받으세요.

In [ ]:
import getpass
from download_training_data import download_training_data
try:
    from google.colab import userdata
    signed_url=userdata.get('FDI_TRAINING_URL')
except Exception:
    signed_url=getpass.getpass('Supabase 서명 URL: ')
download_training_data(signed_url, WORK/'data/raw')
del signed_url
print('학습 자료 12개 다운로드·해시 확인 완료')


In [ ]:
import subprocess,sys,datetime
OUTPUT=WORK/'artifacts'/datetime.datetime.now().strftime('run_%Y%m%d_%H%M%S')
subprocess.run([sys.executable,'run_all.py','--data-dir',str(WORK/'data/raw'),'--output-dir',str(OUTPUT)],check=True)


In [ ]:
import json,pandas as pd
summary=json.loads((OUTPUT/'validation_summary.json').read_text())
display(pd.DataFrame(summary['folds']))
print('108개월 내부 검증 평균 IC:',summary['overall_mean_ic'])
print('산출물 폴더:',OUTPUT)
